In [ ]:
# Combined Part B + Part D baseline: k-fold-by-NPI, same protocol as Notebooks 05 / 08.
# SMOKE=1 (default) = 1 run, 1 fold. sbatch sets SMOKE=0.
# LABEL_COL: fraud | fraud_h2 | fraud_nob4.  LEARNERS: "XGB" (default) or "XGB,RF".
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

SMOKE     = os.environ.get("SMOKE", "1") == "1"
LABEL_COL = os.environ.get("LABEL_COL", "fraud")
LEARNERS  = os.environ.get("LEARNERS", "XGB").split(",")
N_JOBS    = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
BD       = DATA_DIR / "processed/combined/bd_labeled_2019_2023.parquet"
RES_DIR  = DATA_DIR / "results/combined"; RES_DIR.mkdir(parents=True, exist_ok=True)
TAG      = f"nb11_{LABEL_COL}" + ("_smoke" if SMOKE else "")
RES_CSV  = RES_DIR / f"{TAG}_fold_results.csv"
print("SMOKE:", SMOKE, "| label:", LABEL_COL, "| learners:", LEARNERS, "| results →", RES_CSV.name)

## Design

Three feature sets, trained on **all** provider-years, same folds:
- **B** — Part B features + Part B provider type
- **D** — Part D features + Part D provider type
- **BD** — both

All three include `has_partb` / `has_partd`. Each test fold is scored overall and **per group** (`both`, `B_only`, `D_only`). The key comparison is on `both` rows: does seeing billing **and** prescribing beat either one alone on the same providers?

RF is off by default (slow on ~7M rows × ~400 columns); set `LEARNERS=XGB,RF` to add it.

In [ ]:
N_RUNS, N_FOLDS = (1, 5) if SMOKE else (6, 5)
FOLDS_TO_RUN    = [0] if SMOKE else list(range(N_FOLDS))

def make_model(name, seed):
    if name == "XGB":
        return XGBClassifier(max_depth=4, n_jobs=N_JOBS, random_state=seed,
                             tree_method="hist", eval_metric="logloss")
    return RandomForestClassifier(max_depth=16, n_jobs=N_JOBS, random_state=seed)

In [ ]:
df = pd.read_parquet(BD)
LABELS = ["fraud", "fraud_h2", "fraud_nob4"]
df = df[df[LABEL_COL] >= 0].reset_index(drop=True)          # -1 = excluded for this label
print(df.shape, "| fraud rows:", int(df[LABEL_COL].sum()))

B_FEATS = [c for c in df.columns if c.startswith("b_") and c not in ("b_type", "b_ent_cd")]
D_FEATS = [c for c in df.columns if c.startswith("d_") and c != "d_type"]
FLAGS   = ["has_partb", "has_partd"]
print("B feats:", len(B_FEATS), "| D feats:", len(D_FEATS))

b_pt = pd.get_dummies(df["b_type"].where(df.has_partb == 1), prefix="bpt", dtype=np.float32)
d_pt = pd.get_dummies(df["d_type"].where(df.has_partd == 1), prefix="dpt", dtype=np.float32)
FEATS = B_FEATS + D_FEATS + FLAGS + list(b_pt.columns) + list(d_pt.columns)
X_all = np.hstack([df[B_FEATS + D_FEATS + FLAGS].to_numpy(np.float32),
                   b_pt.to_numpy(np.float32), d_pt.to_numpy(np.float32)])
y_all = df[LABEL_COL].to_numpy(np.int8)
grp   = df["group"].to_numpy()
ci_ = {f: i for i, f in enumerate(FEATS)}
DS_COLS = {"B":  [ci_[c] for c in B_FEATS + FLAGS + list(b_pt.columns)],
           "D":  [ci_[c] for c in D_FEATS + FLAGS + list(d_pt.columns)],
           "BD": list(range(len(FEATS)))}
del b_pt, d_pt
for k, v in DS_COLS.items(): print(k, "features:", len(v))
print("X_all:", X_all.shape, f"{X_all.nbytes/1e9:.1f} GB")

In [ ]:
npi = df["npi"].to_numpy()
npi_ever = df.groupby("npi")[LABEL_COL].max()

def assign_folds(seed):
    rng = np.random.default_rng(seed)
    parts = []
    for cls in (0, 1):
        ids = rng.permutation(npi_ever.index[npi_ever == cls].to_numpy())
        parts.append(pd.Series(np.arange(len(ids)) % N_FOLDS, index=ids))
    return pd.Series(npi).map(pd.concat(parts)).to_numpy()

f0 = assign_folds(0)
chk = pd.DataFrame({"npi": npi, "fold": f0, "y": y_all})
assert chk.groupby("npi").fold.nunique().max() == 1, "NPI leaks across folds"
print(chk.groupby("fold").agg(rows=("y", "size"), fraud_rows=("y", "sum")))
del chk

In [ ]:
def evaluate(y_true, p, thr):
    if y_true.min() == y_true.max():
        return None
    pred = (p >= thr).astype(np.int8)
    tp = ((pred == 1) & (y_true == 1)).sum(); fn = ((pred == 0) & (y_true == 1)).sum()
    tn = ((pred == 0) & (y_true == 0)).sum(); fp = ((pred == 1) & (y_true == 0)).sum()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    return {"AUC": roc_auc_score(y_true, p), "AUPRC": average_precision_score(y_true, p),
            "base_rate": y_true.mean(), "TPR": tpr, "TNR": tnr, "GMean": np.sqrt(tpr * tnr),
            "test_pos": int(y_true.sum())}

In [ ]:
done = set()
if RES_CSV.exists():
    prev = pd.read_csv(RES_CSV)
    done = set(zip(prev.run, prev.fold, prev.dataset, prev.learner))
print("Already done:", len(done) // 4)

for run in range(N_RUNS):
    folds = assign_folds(run)
    for k in FOLDS_TO_RUN:
        te = np.where(folds == k)[0]; tr = np.where(folds != k)[0]
        thr = y_all[tr].mean()
        for ds, cols in DS_COLS.items():
            for lr in LEARNERS:
                if (run, k, ds, lr) in done: continue
                t0 = time.time()
                m = make_model(lr, seed=run).fit(X_all[np.ix_(tr, cols)], y_all[tr])
                p = m.predict_proba(X_all[np.ix_(te, cols)])[:, 1]
                sec = round(time.time() - t0, 1)
                rows = []
                for g in ["all", "both", "B_only", "D_only"]:
                    sel = slice(None) if g == "all" else (grp[te] == g)
                    r = evaluate(y_all[te][sel], p[sel], thr)
                    if r: rows.append({"run": run, "fold": k, "dataset": ds, "learner": lr,
                                       "label": LABEL_COL, "group": g, **r, "fit_sec": sec})
                pd.DataFrame(rows).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)
                a = {r["group"]: r["AUC"] for r in rows}
                print(f"run{run} fold{k} {ds:>2} {lr}: " +
                      " ".join(f"{g}={v:.4f}" for g, v in a.items()) + f" ({sec}s)")

In [ ]:
res = pd.read_csv(RES_CSV)
def ci(s):
    m = s.mean(); h = 1.96 * s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan
    return f"{m:.4f} ({m-h:.4f}, {m+h:.4f})"
for metric in ["AUC", "AUPRC"]:
    for lr in LEARNERS:
        t = (res[res.learner == lr]
             .pivot_table(index="dataset", columns="group", values=metric, aggfunc=ci)
             .reindex(index=["B", "D", "BD"], columns=["all", "both", "B_only", "D_only"]))
        print(f"\n{metric} — {lr}, label={LABEL_COL}, mean (95% CI)\n", t.to_string())
        t.to_csv(RES_DIR / f"{TAG}_{lr}_{metric}.csv")
lift = res.assign(lift=res.AUPRC / res.base_rate).pivot_table(index="dataset", columns="group", values="lift")
print("\nAUPRC / base rate (× better than random)\n", lift.round(2).to_string())